In [1]:
# ==========================================
# CELL 1: SETUP & CLEANING
# ==========================================
import pandas as pd
import numpy as np
import torch
import xgboost as xgb
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, accuracy_score
from scipy.special import softmax

from datasets import Dataset
from transformers import (
    AutoTokenizer, AutoModelForSequenceClassification, 
    DataCollatorWithPadding, TrainingArguments, Trainer
)
import evaluate

# 1. Load Data (Replace with your actual paths)
df = pd.read_csv("/Users/Tyler/Documents/Stanford Classes/CS229/CS229Final/Dataframes/tyrm-alladults.csv") 

# 2. Basic Cleaning 
df["chiefcomplaint"] = df["chiefcomplaint"].astype(str).fillna("").str.strip()
df = df[df["chiefcomplaint"].str.len() > 0].copy()

# 3. Format Target (Acuity 1-5 becomes 0-4 for ML models)
df = df[df["acuity"].isin([1, 2, 3, 4, 5])].copy()
df["label"] = df["acuity"].astype(int) - 1  # 'label' is the standard HF name

# 4. Train / Val / Test Split (60% Train, 20% Val, 20% Test)
train_val_df, test_df = train_test_split(df, test_size=0.2, random_state=42, stratify=df["label"])
train_df, val_df = train_test_split(train_val_df, test_size=0.25, random_state=42, stratify=train_val_df["label"])

print(f"Train size: {len(train_df)} | Val size: {len(val_df)} | Test size: {len(test_df)}")

Train size: 267489 | Val size: 89164 | Test size: 89164


In [10]:
print(train_df.shape[0])
print(val_df.shape[0])
print(test_df.shape[0])

sum = train_df.shape[0] + val_df.shape[0] + test_df.shape[0]
print(sum)


267489
89164
89164
445817


In [5]:
# ==========================================
# CELL 2: TABULAR BASELINE (XGBOOST)
# ==========================================
cols_to_drop = ["chiefcomplaint", "acuity", "label", "race"] # Drop text and targets

# Extract features and targets
X_train_tab = train_df.drop(columns=cols_to_drop, errors='ignore')
y_train = train_df["label"].values

X_val_tab = val_df.drop(columns=cols_to_drop, errors='ignore')
y_val = val_df["label"].values

X_test_tab = test_df.drop(columns=cols_to_drop, errors='ignore')
y_test = test_df["label"].values

# Impute and Scale based ONLY on training data
train_mean = X_train_tab.mean()
scaler = StandardScaler()

X_train_tab = scaler.fit_transform(X_train_tab.fillna(train_mean))
X_val_tab = scaler.transform(X_val_tab.fillna(train_mean))
X_test_tab = scaler.transform(X_test_tab.fillna(train_mean))

# Train XGBoost
print("Training XGBoost on Tabular Data...")
xgb_model = xgb.XGBClassifier(
    n_estimators=500, learning_rate=0.05, objective='multi:softprob',
    num_class=5, early_stopping_rounds=25, random_state=42
)

xgb_model.fit(
    X_train_tab, y_train,
    eval_set=[(X_val_tab, y_val)],
    verbose=False
)

# Extract Meta-Features (Probabilities)
print("Extracting Tabular Probabilities...")
tab_probs_train = xgb_model.predict_proba(X_train_tab)
tab_probs_val   = xgb_model.predict_proba(X_val_tab)
tab_probs_test  = xgb_model.predict_proba(X_test_tab)

Training XGBoost on Tabular Data...
Extracting Tabular Probabilities...


In [12]:
# ==========================================
# CELL 3: TEXT BASELINE (Bio_ClinicalBERT)
# ==========================================
model_name = "emilyalsentzer/Bio_ClinicalBERT"
tokenizer = AutoTokenizer.from_pretrained(model_name)

def tokenize(batch):
    return tokenizer(batch["chiefcomplaint"], truncation=True, padding=True, max_length=32)

# Convert Pandas to HF Datasets
train_ds = Dataset.from_pandas(train_df[["chiefcomplaint", "label"]]).map(tokenize, batched=True)
val_ds   = Dataset.from_pandas(val_df[["chiefcomplaint", "label"]]).map(tokenize, batched=True)
test_ds  = Dataset.from_pandas(test_df[["chiefcomplaint", "label"]]).map(tokenize, batched=True)

# Initialize Model and Trainer
model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=5)
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

args = TrainingArguments(
    output_dir="clinicalbert_triage", learning_rate=2e-5,
    per_device_train_batch_size=32, num_train_epochs=2,
    eval_strategy="epoch", save_strategy="epoch",
    load_best_model_at_end=True, report_to="none"
)

trainer = Trainer(
    model=model, args=args,
    train_dataset=train_ds, eval_dataset=val_ds,
    processing_class=tokenizer, data_collator=data_collator
)

print("Fine-tuning Bio_ClinicalBERT...")
trainer.train()

# Extract Meta-Features (Logits -> Softmax -> Probabilities)
print("Extracting Text Probabilities...")
text_logits_train = trainer.predict(train_ds).predictions
text_logits_val   = trainer.predict(val_ds).predictions
text_logits_test  = trainer.predict(test_ds).predictions

text_probs_train = softmax(text_logits_train, axis=1)
text_probs_val   = softmax(text_logits_val, axis=1)
text_probs_test  = softmax(text_logits_test, axis=1)

Map:   0%|          | 0/267489 [00:00<?, ? examples/s]

Map:   0%|          | 0/89164 [00:00<?, ? examples/s]

Map:   0%|          | 0/89164 [00:00<?, ? examples/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: emilyalsentzer/Bio_ClinicalBERT
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.decoder.weight             | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Conside

...

/opt/miniconda3/envs/triage229/lib/python3.11/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['bert.embeddings.LayerNorm.weight', 'bert.embeddings.LayerNorm.bias', 'bert.encoder.layer.0.attention.output.LayerNorm.weight', 'bert.encoder.layer.0.attention.output.LayerNorm.bias', 'bert.encoder.layer.0.output.LayerNorm.weight', 'bert.encoder.layer.0.output.LayerNorm.bias', 'bert.encoder.layer.1.attention.output.LayerNorm.weight', 'bert.encoder.layer.1.attention.output.LayerNorm.bias', 'bert.encoder.layer.1.output.LayerNorm.weight', 'bert.encoder.layer.1.output.LayerNorm.bias', 'bert.encoder.layer.2.attention.output.LayerNorm.weight', 'bert.encoder.layer.2.attention.output.LayerNorm.bias', 'bert.encoder.layer.2.output.LayerNorm.weight', 'bert.encoder.layer.2.output.LayerNorm.bias', 'bert.encoder.layer.3.attention.output.LayerNorm.weight', 'bert.encoder.layer.3.attention.output.LayerNorm.bias', 'bert.encoder.layer.3.output.LayerNorm.weight', 'bert.encoder.layer.3.output.LayerNorm.bias', 'bert.encoder.layer.4.attention.output.La

Extracting Text Probabilities...


/opt/miniconda3/envs/triage229/lib/python3.11/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


In [36]:
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    balanced_accuracy_score,
    confusion_matrix,
    classification_report,
    cohen_kappa_score
)

def evaluate_metrics(y_true, y_pred, label):
    print(f"\n--- {label} ---")
    print("Accuracy:", f"{accuracy_score(y_true, y_pred):.3f}")
    print("Balanced Accuracy:", f"{balanced_accuracy_score(y_true, y_pred):.3f}")
    print("Macro F1:", f"{f1_score(y_true, y_pred, average='macro'):.3f}")
    print("Cohen's Kappa:", f"{cohen_kappa_score(y_true, y_pred):.3f}")

    
    # print("\nConfusion Matrix:")
    # print(confusion_matrix(y_true, y_pred))
    
    # print("\nClassification Report:")
    # print(classification_report(y_true, y_pred))

def evaluate_asymmetric_dropout(drop_rate_text, drop_rate_tab, text_train, tab_train, y_train, text_test, tab_test, y_test):
    # 1. Apply Independent Dropout Masks
    np.random.seed(42)
    n_samples = text_train.shape[0]
    
    dropped_text = text_train.copy()
    dropped_tab  = tab_train.copy()
    
    # Roll one random number per patient
    rand_rolls = np.random.rand(n_samples)
    
    # Drop Text (e.g., if 0.40, rolls from 0.00 to 0.40 drop text)
    drop_text_mask = rand_rolls < drop_rate_text
    dropped_text[drop_text_mask] = 0.0
    
    # Drop Tabular (e.g., if 0.10, rolls from 0.40 to 0.50 drop tabular)
    # This prevents overlap, ensuring we never drop BOTH at once.
    drop_tab_mask = (rand_rolls >= drop_rate_text) & (rand_rolls < (drop_rate_text + drop_rate_tab))
    dropped_tab[drop_tab_mask] = 0.0

    # 2. Stack and Train Meta-Classifier
    X_train_meta = np.hstack((dropped_text, dropped_tab))
    meta_clf = LogisticRegression(max_iter=1000, random_state=42)
    meta_clf.fit(X_train_meta, y_train)
    
    pred_both = meta_clf.predict(np.hstack((text_test, tab_test)))
    pred_no_tab = meta_clf.predict(np.hstack((text_test, np.zeros_like(tab_test))))
    pred_no_text = meta_clf.predict(np.hstack((np.zeros_like(text_test), tab_test)))

    print(f"\n======== Dropout Training Rates ========")
    print(f"Text: {drop_rate_text*100:.0f}% | Tabular: {drop_rate_tab*100:.0f}%")

    evaluate_metrics(y_test, pred_both, "Both Modalities Intact")
    evaluate_metrics(y_test, pred_no_tab, "Text Only (Tabular Dropped)")
    evaluate_metrics(y_test, pred_no_text, "Tabular Only (Text Dropped)")
    
    # # 3. Evaluate Robustness on Test Set (The Extreme Scenarios)
    # acc_perfect = accuracy_score(y_test, meta_clf.predict(np.hstack((text_test, tab_test))))
    # acc_no_tab  = accuracy_score(y_test, meta_clf.predict(np.hstack((text_test, np.zeros_like(tab_test)))))
    # acc_no_text = accuracy_score(y_test, meta_clf.predict(np.hstack((np.zeros_like(text_test), tab_test))))
    
    # print(f"\n--- Dropout Training Rates -> Text: {drop_rate_text*100}%, Tab: {drop_rate_tab*100}% ---")
    # print(f"Test Accuracy (Both Intact):  {acc_perfect:.3f}")
    # print(f"Test Accuracy (No Tabular):   {acc_no_tab:.3f}")
    # print(f"Test Accuracy (No Text):      {acc_no_text:.3f}")
    
    return meta_clf

In [48]:
# ==========================================
# Run the experiments with Asymmetric Pairs
# ==========================================
dropout_pairs = [
    (0.00, 0.00), # Baseline: No dropout
	(0.05, 0.05),
	(0.10, 0.10),
    (0.15, 0.15), # Balanced dropout (30% total)
	(0.20, 0.20),
	(0.30, 0.30),
	(0.40, 0.40),
	(0.50, 0.50),
	(0.60, 0.60),
    (0.40, 0.05), # Handicap Text: Force model to learn from tabular
    (0.05, 0.40),  # Handicap Tabular: Force model to learn from text
	(0.00, 0.50),
	(0.50, 0.00),
]

meta_X_val_text = text_probs_val
meta_X_val_tab  = tab_probs_val
meta_y_val      = y_val

meta_X_test_text = text_probs_test
meta_X_test_tab  = tab_probs_test
meta_y_test      = y_test

trained_meta_models = {}

print("\n" + "="*50)
print(" EVALUATING ROBUSTNESS ON ADULT COHORT")
print("="*50)
for drop_text, drop_tab in dropout_pairs:
    clf = evaluate_asymmetric_dropout(
        drop_text, drop_tab,
        meta_X_val_text, meta_X_val_tab, meta_y_val,
        meta_X_test_text, meta_X_test_tab, meta_y_test
    )
    trained_meta_models[(drop_text, drop_tab)] = clf


# meta_X_train_text = np.vstack((text_probs_train, text_probs_val))
# meta_X_train_tab  = np.vstack((tab_probs_train, tab_probs_val))
# meta_y_train      = np.concatenate((y_train, y_val))

# meta_X_test_text = text_probs_test
# meta_X_test_tab  = tab_probs_test
# meta_y_test      = y_test

# trained_meta_models = {}

# print("\n" + "="*50)
# print(" EVALUATING ROBUSTNESS ON ADULT COHORT")
# print("="*50)
# for drop_text, drop_tab in dropout_pairs:
# 	clf = evaluate_asymmetric_dropout(
# 		drop_text, drop_tab,
# 		meta_X_train_text, meta_X_train_tab, meta_y_train,
# 		meta_X_test_text, meta_X_test_tab, meta_y_test
# 	)
# 	trained_meta_models[(drop_text, drop_tab)] = clf


 EVALUATING ROBUSTNESS ON ADULT COHORT

======== Dropout Training Rates ========
Text: 0% | Tabular: 0%

--- Both Modalities Intact ---
Accuracy: 0.696
Balanced Accuracy: 0.471
Macro F1: 0.497
Cohen's Kappa: 0.466

--- Text Only (Tabular Dropped) ---
Accuracy: 0.655
Balanced Accuracy: 0.461
Macro F1: 0.478
Cohen's Kappa: 0.413

--- Tabular Only (Text Dropped) ---
Accuracy: 0.599
Balanced Accuracy: 0.372
Macro F1: 0.377
Cohen's Kappa: 0.247

======== Dropout Training Rates ========
Text: 5% | Tabular: 5%

--- Both Modalities Intact ---
Accuracy: 0.696
Balanced Accuracy: 0.471
Macro F1: 0.497
Cohen's Kappa: 0.466

--- Text Only (Tabular Dropped) ---
Accuracy: 0.667
Balanced Accuracy: 0.372
Macro F1: 0.396
Cohen's Kappa: 0.400

--- Tabular Only (Text Dropped) ---
Accuracy: 0.598
Balanced Accuracy: 0.348
Macro F1: 0.346
Cohen's Kappa: 0.210

======== Dropout Training Rates ========
Text: 10% | Tabular: 10%

--- Both Modalities Intact ---
Accuracy: 0.696
Balanced Accuracy: 0.472
Macro F1: 

In [45]:
# ==========================================
# CELL 5: OUT-OF-DISTRIBUTION EVALUATION (PEDIATRIC)
# ==========================================
print("\n" + "="*50)
print(" PREPARING PEDIATRIC DATASET (EXTERNAL VALIDATION)")
print("="*50)

# 1. Load and clean pediatric data
peds_path = '/Users/Tyler/Documents/Stanford Classes/CS229/CS229Final/Dataframes/tyrm-pedsNHAMCS.csv'
peds_df = pd.read_csv(peds_path)

# IMPORTANT: fillna BEFORE astype(str), otherwise NaN becomes the string "nan"
peds_df["chiefcomplaint"] = peds_df["chiefcomplaint"].fillna("").astype(str).str.strip()

# Keep only non-empty complaints
peds_df = peds_df[peds_df["chiefcomplaint"].str.len() > 0].copy()

# Keep only valid acuity labels
peds_df = peds_df[peds_df["acuity"].isin([1, 2, 3, 4, 5])].copy()
peds_df["label"] = peds_df["acuity"].astype(int) - 1

meta_y_peds = peds_df["label"].values

print(f"Pediatric cohort size after filtering: {len(peds_df)}")

# 2. Process tabular data
# Drop the same columns you dropped during adult training
X_peds_tab = peds_df.drop(columns=cols_to_drop, errors="ignore")

# CRITICAL:
# Reindex to the exact feature order used during training.
# Replace `tab_feature_names` with whatever variable you used to store
# the adult training tabular column names.

# Impute using ADULT training means, then scale using ADULT scaler
X_peds_tab_imputed = X_peds_tab.fillna(train_mean)
X_peds_tab_scaled = scaler.transform(X_peds_tab_imputed)

print("Extracting Peds Tabular Probabilities (XGBoost)...")
meta_X_peds_tab = xgb_model.predict_proba(X_peds_tab_scaled)

# 3. Process text data
# Match your earlier val/test pipeline as closely as possible
peds_ds = Dataset.from_pandas(peds_df[["chiefcomplaint", "label"]])
peds_ds = peds_ds.map(tokenize, batched=True, remove_columns=["chiefcomplaint"])

print("Extracting Peds Text Probabilities (Bio_ClinicalBERT)...")
peds_logits = trainer.predict(peds_ds).predictions
meta_X_peds_text = softmax(peds_logits, axis=1)




 PREPARING PEDIATRIC DATASET (EXTERNAL VALIDATION)
Pediatric cohort size after filtering: 5581
Extracting Peds Tabular Probabilities (XGBoost)...


Map:   0%|          | 0/5581 [00:00<?, ? examples/s]

Extracting Peds Text Probabilities (Bio_ClinicalBERT)...


/opt/miniconda3/envs/triage229/lib/python3.11/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


In [49]:
# ==========================================
# 4. Evaluate all models on Pediatric Data
# ==========================================
print("\n" + "="*50)
print(" EVALUATING ROBUSTNESS ON PEDIATRIC COHORT")
print("="*50)

peds_results = []

for (drop_text, drop_tab), meta_clf in trained_meta_models.items():

    # Both modalities
    X_peds_both = np.hstack((meta_X_peds_text, meta_X_peds_tab))
    pred_both = meta_clf.predict(X_peds_both)

    # Text only
    X_peds_no_tab = np.hstack((meta_X_peds_text, np.zeros_like(meta_X_peds_tab)))
    pred_no_tab = meta_clf.predict(X_peds_no_tab)

    # Tabular only
    X_peds_no_text = np.hstack((np.zeros_like(meta_X_peds_text), meta_X_peds_tab))
    pred_no_text = meta_clf.predict(X_peds_no_text)

    print(f"\n{'='*60}")
    print(f"Dropout Training Rates -> Text: {drop_text*100:.0f}%, Tab: {drop_tab*100:.0f}%")
    print(f"{'='*60}")

    evaluate_metrics(meta_y_peds, pred_both, "Peds: Both Modalities Intact")
    evaluate_metrics(meta_y_peds, pred_no_tab, "Peds: Text Only (Tabular Dropped)")
    evaluate_metrics(meta_y_peds, pred_no_text, "Peds: Tabular Only (Text Dropped)")


# print("\n" + "="*50)
# print(" EVALUATING ROBUSTNESS ON PEDIATRIC COHORT")
# print("="*50)

# for (drop_text, drop_tab), meta_clf in trained_meta_models.items():
    
#     # 1. Both Modalities Intact
#     X_peds_perfect = np.hstack((meta_X_peds_text, meta_X_peds_tab))
#     acc_perfect = accuracy_score(meta_y_peds, meta_clf.predict(X_peds_perfect))
    
#     # 2. No Tabular (Simulating missing vitals)
#     X_peds_no_tab = np.hstack((meta_X_peds_text, np.zeros_like(meta_X_peds_tab)))
#     acc_no_tab = accuracy_score(meta_y_peds, meta_clf.predict(X_peds_no_tab))
    
#     # 3. No Text (Simulating missing chief complaint)
#     X_peds_no_text = np.hstack((np.zeros_like(meta_X_peds_text), meta_X_peds_tab))
#     acc_no_text = accuracy_score(meta_y_peds, meta_clf.predict(X_peds_no_text))
    
#     print(f"\n--- Dropout Training Rates -> Text: {drop_text*100}%, Tab: {drop_tab*100}% ---")
#     print(f"Peds Accuracy (Both Intact):  {acc_perfect:.3f}")
#     print(f"Peds Accuracy (No Tabular):   {acc_no_tab:.3f}")
#     print(f"Peds Accuracy (No Text):      {acc_no_text:.3f}")


 EVALUATING ROBUSTNESS ON PEDIATRIC COHORT

Dropout Training Rates -> Text: 0%, Tab: 0%

--- Peds: Both Modalities Intact ---
Accuracy: 0.561
Balanced Accuracy: 0.307
Macro F1: 0.314
Cohen's Kappa: 0.260

--- Peds: Text Only (Tabular Dropped) ---
Accuracy: 0.515
Balanced Accuracy: 0.300
Macro F1: 0.309
Cohen's Kappa: 0.206

--- Peds: Tabular Only (Text Dropped) ---
Accuracy: 0.521
Balanced Accuracy: 0.284
Macro F1: 0.287
Cohen's Kappa: 0.177

Dropout Training Rates -> Text: 5%, Tab: 5%

--- Peds: Both Modalities Intact ---
Accuracy: 0.563
Balanced Accuracy: 0.308
Macro F1: 0.314
Cohen's Kappa: 0.262

--- Peds: Text Only (Tabular Dropped) ---
Accuracy: 0.499
Balanced Accuracy: 0.288
Macro F1: 0.288
Cohen's Kappa: 0.182

--- Peds: Tabular Only (Text Dropped) ---
Accuracy: 0.524
Balanced Accuracy: 0.272
Macro F1: 0.261
Cohen's Kappa: 0.204

Dropout Training Rates -> Text: 10%, Tab: 10%

--- Peds: Both Modalities Intact ---
Accuracy: 0.564
Balanced Accuracy: 0.308
Macro F1: 0.314
Cohen's 

In [31]:
peds_df

,race,gender,age_at_visit,temperature,heartrate,resprate,sbp,dbp,o2sat,pain,unable,chiefcomplaint,acuity,label
0,Black/African American Only,0,8,97.5,105.0,22.0,113.0,78.0,100.0,0.0,0,Migraine headache,4,3
1,Blank,1,15,98.7,80.0,22.0,137.0,80.0,100.0,0.0,0,Behavioral disturbances,3,2
2,Black/African American Only,0,6,97.1,115.0,23.0,NaN,NaN,98.0,NaN,1,Other musculoskeletal symptoms,4,3
3,Black/African American Only,1,0,100.8,158.0,NaN,NaN,NaN,100.0,NaN,1,Fever,3,2
4,Blank,1,4,98.1,142.0,24.0,NaN,NaN,100.0,NaN,1,"Burn, all degrees, to extremities",3,2
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5576,Asian Only,0,16,97.7,98.0,22.0,146.0,122.0,98.0,NaN,1,"Injury, other and unspecified of head...",1,0
5577,More than one race reported,1,10,97.5,104.0,21.0,116.0,94.0,100.0,8.0,0,"Injury, other and unspecified of head...",1,0
5578,Blank,0,0,99.9,154.0,30.0,NaN,NaN,98.0,NaN,1,Sneezing,3,2
5579,Blank,0,15,99.0,69.0,18.0,133.0,86.0,99.0,0.0,0,Depression,5,4


In [32]:
import numpy as np
from sklearn.metrics import accuracy_score

# ==========================================
# ERROR ANALYSIS: AGE STRATIFICATION
# ==========================================
print("\n" + "="*50)
print(" ERROR ANALYSIS: AGE STRATIFICATION (5% DROPOUT)")
print("="*50)

# Grab the best performing meta-classifier
best_clf = trained_meta_models[(0.05, 0.05)]

# Define standard clinical pediatric age brackets
age_brackets = {
    "Infants (0-1 years)": (0, 1),
    "Toddlers/Preschool (2-5 years)": (2, 5),
    "School Age (6-12 years)": (6, 12),
    "Adolescents (13-17 years)": (13, 17)
}

# The name of the age column in your peds_df
age_col = 'age_at_visit' 

for bracket_name, (age_min, age_max) in age_brackets.items():
    # 1. Create a boolean mask to filter the dataframe for this specific age group
    mask = (peds_df[age_col] >= age_min) & (peds_df[age_col] <= age_max)
    
    n_patients = mask.sum()
    if n_patients == 0:
        print(f"\n--- {bracket_name} ---")
        print("No patients found in this age range.")
        continue
        
    # 2. Apply the mask to our labels and meta-features
    y_true_bracket = meta_y_peds[mask]
    X_text_bracket = meta_X_peds_text[mask]
    X_tab_bracket  = meta_X_peds_tab[mask]
    
    # 3. Evaluate: Both Modalities Intact
    X_perfect = np.hstack((X_text_bracket, X_tab_bracket))
    acc_perfect = accuracy_score(y_true_bracket, best_clf.predict(X_perfect))
    
    # 4. Evaluate: No Tabular (Simulating missing vitals)
    X_no_tab = np.hstack((X_text_bracket, np.zeros_like(X_tab_bracket)))
    acc_no_tab = accuracy_score(y_true_bracket, best_clf.predict(X_no_tab))
    
    # 5. Evaluate: No Text (Simulating missing chief complaint)
    X_no_text = np.hstack((np.zeros_like(X_text_bracket), X_tab_bracket))
    acc_no_text = accuracy_score(y_true_bracket, best_clf.predict(X_no_text))
    
    # 6. Print Results
    print(f"\n--- {bracket_name} (n={n_patients}) ---")
    print(f"Accuracy (Both Intact):  {acc_perfect:.3f}")
    print(f"Accuracy (No Tabular):   {acc_no_tab:.3f}")
    print(f"Accuracy (No Text):      {acc_no_text:.3f}")


 ERROR ANALYSIS: AGE STRATIFICATION (5% DROPOUT)

--- Infants (0-1 years) (n=1181) ---
Accuracy (Both Intact):  0.524
Accuracy (No Tabular):   0.450
Accuracy (No Text):      0.531

--- Toddlers/Preschool (2-5 years) (n=1344) ---
Accuracy (Both Intact):  0.546
Accuracy (No Tabular):   0.460
Accuracy (No Text):      0.556

--- School Age (6-12 years) (n=1572) ---
Accuracy (Both Intact):  0.565
Accuracy (No Tabular):   0.508
Accuracy (No Text):      0.532

--- Adolescents (13-17 years) (n=1484) ---
Accuracy (Both Intact):  0.606
Accuracy (No Tabular):   0.564
Accuracy (No Text):      0.481
